# Exercício 04 — PySpark: Joins entre DataFrames

**Tópico:** PySpark — joins (inner, left, right, full)

---

## Setup
Faça o upload dos arquivos `clientes.csv`, `pedidos.csv` e `funcionarios.csv` para o DBFS.  
Atualize os caminhos abaixo.

---

## Exercício 1 — Inner join
Leia os arquivos `clientes.csv` e `pedidos.csv`.  
Faça um **inner join** entre eles usando a coluna `cliente_id` (em pedidos) e `id` (em clientes).  
Exiba: `nome` do cliente, `produto`, `valor_total` e `data` do pedido.

---

## Exercício 2 — Left join
Faça um **left join** de `clientes` com `pedidos`.  
O resultado deve mostrar **todos os clientes**, mesmo os que ainda não fizeram pedidos.  
Quantos clientes não têm nenhum pedido?

---

## Exercício 3 — Aggregação após join
Usando o resultado do inner join do exercício 1:  
- Agrupe por `nome` do cliente
- Calcule o **total gasto** (`sum` de `valor_total`) e a **quantidade de pedidos** (`count`)
- Ordene pelo total gasto de forma decrescente

---

## Exercício 4 — Join com expressão
Leia `funcionarios.csv` e `pedidos.csv`.  
Faça um join entre eles onde `funcionarios.nome` contém o nome do `pedidos.vendedor`... mas espera — essa coluna não existe em pedidos!  
Então: leia `vendas.csv` e faça um join de `funcionarios` com `vendas` onde `vendedor == nome`.  
Exiba `nome`, `departamento`, `produto` e `valor`.

---

## Exercício 5 — Union de DataFrames
Crie dois DataFrames filtrados de `pedidos.csv`:  
- `df_entregues`: apenas pedidos com `status == 'Entregue'`  
- `df_pendentes`: apenas pedidos com `status == 'Pendente'`  

Use `.union()` para combiná-los e conte quantas linhas tem no total.  
Compare com o total de pedidos sem o filtro `Em trânsito`.


In [0]:
CLIENTES_PATH  = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/clientes.csv"
PEDIDOS_PATH   = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/pedidos.csv"
FUNCIONARIOS_PATH = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/funcionarios.csv"
VENDAS_PATH    = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/vendas.csv"

In [0]:
# Exercício 1
clientes_df = spark.read.csv(CLIENTES_PATH, header=True, inferSchema=True)
pedidos_df = spark.read.csv(PEDIDOS_PATH, header=True, inferSchema=True)

df1_join = clientes_df.join(pedidos_df, clientes_df.id == pedidos_df.cliente_id, how="inner")

display(df1_join.orderBy("cliente_id"))


In [0]:
# Exercício 2

clientes_df = spark.read.csv(CLIENTES_PATH, header=True, inferSchema=True)
pedidos_df = spark.read.csv(PEDIDOS_PATH, header=True, inferSchema=True)

df_2_left_join = clientes_df.join(pedidos_df, pedidos_df.cliente_id == clientes_df.id, how="left")
display(df_2_left_join)

In [0]:
# Exercício 3
from pyspark.sql.functions import col, sum, count, desc
df_3 = df1_join
df_3 = df_3.groupBy(col("nome"))
display(df_3.agg(sum("valor_total").alias("total gasto"), count("*").alias("quantidade_pedidos") ).orderBy("total gasto", ascending=False))

In [0]:
# Exercício 4
from pyspark.sql.functions import col
funcionarios_df = spark.read.csv(FUNCIONARIOS_PATH, header=True, inferSchema=True)
pedidos_df = spark.read.csv(PEDIDOS_PATH, header=True, inferSchema=True)
vendas_df = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)
display(funcionarios_df)
df_4_join = vendas_df.join(funcionarios_df, funcionarios_df.nome.contains(vendas_df.vendedor) , how="inner")
display(df_4_join)

In [0]:
# Exercício 5
from pyspark.sql.functions import col
arquivo = spark.read.csv(PEDIDOS_PATH, header=True, inferSchema=True)
df_entregues = arquivo.filter(col("status") == "Entregue")
df_pendentes = arquivo.filter(col("status") == "Pendente")

df_completo = df_entregues.union(df_pendentes)
rows_df_completo = df_completo.count()

df_em_transito = arquivo.filter(col("status") != "Em trânsito")
rows_diferente_em_transito = df_em_transito.count()

print(f"Pendentes e Entregues: {rows_df_completo}" )
print(f"Diferente de 'Em trânsito': {rows_diferente_em_transito}")


